In [22]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

C:\Users\Steven Tsoukalas\Documents\nyc-building-energy


In [23]:
import pandas as pd
import numpy as np

raw = pd.read_csv("data/raw/ll84.csv", low_memory=False)
print(raw.shape)
for c in raw.columns:
    print(c)

(103259, 265)
Calendar Year
Property ID
Property Name
Parent Property ID
Parent Property Name
Year Ending
NYC Borough, Block and Lot (BBL)
NYC Building Identification Number (BIN)
Address 1
City
Postal Code
Primary Property Type - Self Selected
Primary Property Type - Portfolio Manager-Calculated
National Median Reference Property Type
List of All Property Use Types (GFA) (ft²)
Largest Property Use Type
Largest Property Use Type - Gross Floor Area (ft²)
2nd Largest Property Use Type
2nd Largest Property Use Type - Gross Floor Area (ft²)
3rd Largest Property Use Type
3rd Largest Property Use Type - Gross Floor Area (ft²)
Year Built
Construction Status
Number of Buildings
Occupancy
Metered Areas (Energy)
Metered Areas (Water)
ENERGY STAR Score
National Median ENERGY STAR Score
Target ENERGY STAR Score
Reason(s) for No Score
ENERGY STAR Certification - Year(s) Certified (Score)
Eligible for Certification for Report PED (Y/N)
ENERGY STAR Certification - Last Approval Date
ENERGY STAR Certi

In [24]:
import pandas as pd
import numpy as np

raw = pd.read_csv("data/raw/ll84.csv", low_memory=False)
print(raw.shape)
for c in raw.columns:
    print(c)
    

(103259, 265)
Calendar Year
Property ID
Property Name
Parent Property ID
Parent Property Name
Year Ending
NYC Borough, Block and Lot (BBL)
NYC Building Identification Number (BIN)
Address 1
City
Postal Code
Primary Property Type - Self Selected
Primary Property Type - Portfolio Manager-Calculated
National Median Reference Property Type
List of All Property Use Types (GFA) (ft²)
Largest Property Use Type
Largest Property Use Type - Gross Floor Area (ft²)
2nd Largest Property Use Type
2nd Largest Property Use Type - Gross Floor Area (ft²)
3rd Largest Property Use Type
3rd Largest Property Use Type - Gross Floor Area (ft²)
Year Built
Construction Status
Number of Buildings
Occupancy
Metered Areas (Energy)
Metered Areas (Water)
ENERGY STAR Score
National Median ENERGY STAR Score
Target ENERGY STAR Score
Reason(s) for No Score
ENERGY STAR Certification - Year(s) Certified (Score)
Eligible for Certification for Report PED (Y/N)
ENERGY STAR Certification - Last Approval Date
ENERGY STAR Certi

In [25]:
cols = {
    "Property ID": "property_id",
    "Property Name": "name",
    "Calendar Year": "year",
    "NYC Borough, Block and Lot (BBL)": "bbl",
    "Borough": "borough",
    "Postal Code": "zip",
    "Primary Property Type - Self Selected": "ptype",
    "Year Built": "year_built",
    "Property GFA - Self-Reported (ft²)": "gfa",
    "Occupancy": "occupancy",
    "Number of Buildings": "n_buildings",
    "ENERGY STAR Score": "energy_star",
    "Site EUI (kBtu/ft²)": "site_eui",
    "Total (Location-Based) GHG Emissions (Metric Tons CO2e)": "ghg",
    "Multifamily Housing - Total Number of Residential Living Units": "units",
    "Latitude": "lat",
    "Longitude": "lon",
    "Neighborhood Tabulation Area (NTA) (2020)": "nta",
}
df = raw[list(cols)].rename(columns=cols)

In [26]:
num = ["year", "year_built", "gfa", "occupancy", "n_buildings",
       "energy_star", "site_eui", "ghg", "units", "lat", "lon"]
for c in num:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ""), errors="coerce")
df["zip"] = df["zip"].astype(str).str[:5]
df["bbl"] = pd.to_numeric(df["bbl"], errors="coerce").astype("Int64")

boroughs = {"1": "MANHATTAN", "2": "BRONX", "3": "BROOKLYN", "4": "QUEENS", "5": "STATEN ISLAND"}
df["borough"] = df["borough"].fillna(df["bbl"].astype(str).str[0].map(boroughs))

In [27]:
print(df["year"].value_counts().sort_index())

year
2022    30485
2023    33684
2024    39090
Name: count, dtype: int64


In [28]:
print(df["year"].value_counts().sort_index())   # rows per reporting year

log = [("start", len(df))]
def step(name, mask):
    global df
    df = df[mask].copy()
    log.append((name, len(df)))

step("latest year", df["year"] == df["year"].max())
step("multifamily only", df["ptype"] == "Multifamily Housing")
step("single building", df["n_buildings"].fillna(1) == 1)
step("not a campus", ~df["name"].str.contains(r"bldgs|sites|child|parent", case=False, na=False))
step("has BBL", df["bbl"].notna())
step("EUI 5–250", df["site_eui"].between(5, 250))
step("occupied", df["occupancy"] > 0)
step("built 1800–2022", df["year_built"].between(1800, 2022))   # newer ones may not be fully occupied yet
step("has floor area", df["gfa"] > 0)
step("one row per BBL", ~df.duplicated("bbl", keep=False))

print(pd.DataFrame(log, columns=["step", "rows"]))
df.to_parquet("data/clean/ll84_clean.parquet")

year
2022    30485
2023    33684
2024    39090
Name: count, dtype: int64
                step    rows
0              start  103259
1        latest year   39090
2   multifamily only   26745
3    single building   24517
4       not a campus   22599
5            has BBL   21859
6          EUI 5–250   17593
7           occupied   17556
8    built 1800–2022   17458
9     has floor area   17458
10   one row per BBL   14563


In [29]:
  chk = raw[(raw["Calendar Year"] == 2024) &
            (raw["Primary Property Type - Self Selected"] == "Multifamily Housing")]
  eui = pd.to_numeric(chk["Site EUI (kBtu/ft²)"], errors="coerce")
  print("missing:", eui.isna().sum(), "| below 5:", (eui < 5).sum(), "| above 500:", (eui > 500).sum())

missing: 6307 | below 5: 195 | above 500: 134


Of 26,745 multifamily buildings in 2024, 6,307 had no energy rating and 329 fell outside 5–500 kBtu/ft^2. Tax lots with more than one submission were dropped. Final: 14,852 buildings.
